# Automatic Number Plate Recognition: YOLOv9 + EasyOCR

Trains a **YOLOv9 (GELAN-C)** license-plate detector, validates it, then reads plates with **EasyOCR** on test images and a traffic video.

Runs on **Kaggle** or **Google Colab** with any accelerator:

| Accelerator | What happens |
|---|---|
| GPU (T4 / P100) | Full training at 640 px (fastest) |
| TPU (Kaggle TPU v5e-8, Colab TPU) | YOLOv9 is plain PyTorch and can't use the TPU chips, so it trains on the TPU machine's large host CPU at 640 px |
| CPU only | 416 px with a frozen backbone so an epoch stays under ~40 min |

**Kaggle:** add the datasets `ronakgohil/license-plate-dataset` and `farzadnekouei/license-plate-recognition-for-red-light-violation`, turn Internet on, pick an accelerator, *Run All*.

**Colab:** add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the 🔑 Secrets panel (the datasets are downloaded with the Kaggle API), pick a runtime, *Run all*.

Results (weights, plots, annotated images, demo video/GIF, CSVs) end up in `out/`.

## 1. Settings

In [ ]:
EPOCHS = None          # None = automatic (40 on GPU, 25 on TPU host CPU, 15 on a small CPU)
SUBSET = 0             # >0: use only this many training images (quick pipeline check)
TRAIN_HOURS = None     # None = automatic (6 h, or 10 h on a small CPU); training stops early and keeps best.pt
VIDEO_SECONDS = 20     # length of the traffic clip used for the video demo

## 2. Environment

In [ ]:
import glob, json, os, random, re, shutil, subprocess, sys

ON_KAGGLE = os.path.exists('/kaggle/input')
ON_COLAB = 'google.colab' in sys.modules
W = '/kaggle/working' if ON_KAGGLE else ('/content' if ON_COLAB else os.getcwd())
OUT, Y9 = f'{W}/out', f'{W}/yolov9'
os.makedirs(OUT, exist_ok=True)
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'  # yolov9 checkpoints pickle full models (torch>=2.6 default changed)


def sh(cmd, cwd=None, check=True):
    print(f'\n$ {cmd}', flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and r.returncode:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.returncode


try:
    import torch
except ImportError:  # some TPU images ship without PyTorch
    sh('pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu')
    import torch

GPU = torch.cuda.is_available()
DEVICE = '0' if GPU else 'cpu'
CPUS = os.cpu_count()
torch.set_num_threads(CPUS)
IMG, FREEZE = 640, 0
if GPU:
    epochs, hours = 40, 6
elif CPUS >= 16:  # TPU VM host
    epochs, hours = 25, 6
else:             # plain CPU session
    IMG, FREEZE, epochs, hours = 416, 10, 15, 10
EPOCHS = EPOCHS or epochs
TRAIN_TIMEOUT = int((TRAIN_HOURS or hours) * 3600)
print(f'platform={"kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local"} torch={torch.__version__} '
      f'cuda={GPU} cpus={CPUS} | epochs={EPOCHS} img={IMG} freeze={FREEZE} timeout={TRAIN_TIMEOUT}s')
if GPU:
    sh('nvidia-smi')

## 3. Datasets

In [ ]:
INPUT = '/kaggle/input'
if not ON_KAGGLE:
    # Colab / local: download the same Kaggle datasets with the Kaggle API
    if ON_COLAB:
        from google.colab import userdata
        for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
            os.environ[k] = userdata.get(k)
    INPUT = f'{W}/input'
    sh('pip install -q kaggle')
    for ref in ('ronakgohil/license-plate-dataset',
                'farzadnekouei/license-plate-recognition-for-red-light-violation'):
        sh(f'kaggle datasets download -q {ref} --unzip -p {INPUT}/{ref.split("/")[1]}')


def find_input(name):
    hits = glob.glob(f'{INPUT}/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f'{name} not found under {INPUT}')
    return hits[0]


root = os.path.dirname(find_input('dataset.yaml'))
for split in ('train', 'val'):
    print(split, len(os.listdir(f'{root}/images/{split}')), 'images,', len(os.listdir(f'{root}/labels/{split}')), 'labels')

if SUBSET:
    # YOLO finds labels by swapping /images/ for /labels/, so mirror both trees with symlinks
    for split in ('train', 'val'):
        os.makedirs(f'{W}/subset/images/{split}', exist_ok=True)
        os.makedirs(f'{W}/subset/labels/{split}', exist_ok=True)
        for img in sorted(os.listdir(f'{root}/images/{split}'))[:SUBSET if split == 'train' else max(SUBSET // 4, 1)]:
            os.symlink(f'{root}/images/{split}/{img}', f'{W}/subset/images/{split}/{img}')
            lbl = os.path.splitext(img)[0] + '.txt'
            if os.path.exists(f'{root}/labels/{split}/{lbl}'):
                os.symlink(f'{root}/labels/{split}/{lbl}', f'{W}/subset/labels/{split}/{lbl}')
    root = f'{W}/subset'

data_yaml = f'{W}/plates.yaml'
with open(data_yaml, 'w') as f:
    f.write(f"train: {root}/images/train\nval: {root}/images/val\nnc: 1\nnames: ['license_plate']\n")

## 4. Install YOLOv9 + EasyOCR

In [ ]:
sh('which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)', check=False)
if not os.path.exists(Y9):
    sh(f'git clone -q https://github.com/SkalskiP/yolov9.git {Y9}')
sh('pip install -q easyocr thop seaborn')
if not os.path.exists(f'{W}/weights/gelan-c.pt'):
    sh(f'wget -q -P {W}/weights https://github.com/WongKinYiu/yolov9/releases/download/v0.1/gelan-c.pt')

# Patch yolov9 for current Pillow / NumPy (removed APIs)
for f in glob.glob(f'{Y9}/**/*.py', recursive=True):
    s = open(f).read()
    n = s.replace('self.font.getsize(label)', 'self.font.getbbox(label)[2:]')
    for t in ('int', 'float', 'bool'):
        n = re.sub(rf'\bnp\.{t}\b', t, n)
    if n != s:
        open(f, 'w').write(n)
        print('patched', f)

## 5. ANPR script (`main.py`)

Same file as `main.py` in the repo: YOLOv9 detection, then EasyOCR on each padded, grayscale, upscaled plate crop. Saved into the YOLOv9 folder as `anpr.py`.

In [ ]:
%%writefile {Y9}/anpr.py
import argparse
import os
import platform
import sys
from pathlib import Path

import torch

FILE = Path(__file__).resolve()
ROOT = FILE.parents[0]  # YOLO root directory
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))  # add ROOT to PATH
if str(ROOT / 'yolov9') not in sys.path:
    sys.path.append(str(ROOT / 'yolov9'))  # add yolov9 to PATH
ROOT = Path(os.path.relpath(ROOT, Path.cwd()))  # relative

from models.common import DetectMultiBackend
from utils.dataloaders import IMG_FORMATS, VID_FORMATS, LoadImages, LoadScreenshots, LoadStreams
from utils.general import (LOGGER, Profile, check_file, check_img_size, check_imshow, check_requirements, colorstr, cv2,
                           increment_path, non_max_suppression, print_args, scale_boxes, strip_optimizer, xyxy2xywh)
from utils.plots import Annotator, colors, save_one_box
from utils.torch_utils import select_device, smart_inference_mode



import csv
import re

import easyocr

OCR_ALLOWLIST = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
_reader = None


def get_reader():
    # Build EasyOCR lazily so `--help` stays fast; fall back to CPU when no GPU is present
    global _reader
    if _reader is None:
        _reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
    return _reader


def perform_ocr_on_image(img, coordinates, pad=0.05):
    """Crop a detected plate from a BGR image and return (text, confidence)."""
    x1, y1, x2, y2 = map(int, coordinates)
    h, w = img.shape[:2]
    px, py = int((x2 - x1) * pad), int((y2 - y1) * pad)
    x1, y1, x2, y2 = max(0, x1 - px), max(0, y1 - py), min(w, x2 + px), min(h, y2 + py)
    cropped_img = img[y1:y2, x1:x2]
    if cropped_img.size == 0:
        return '', 0.0

    gray_img = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2GRAY)
    if gray_img.shape[0] < 64:  # small plates read much better after upscaling
        scale = 64 / gray_img.shape[0]
        gray_img = cv2.resize(gray_img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    results = get_reader().readtext(gray_img, allowlist=OCR_ALLOWLIST, paragraph=False)

    # Plates can be split over two lines, so join the confident pieces top-to-bottom, left-to-right
    results = [r for r in results if r[2] > 0.1]
    results.sort(key=lambda r: (round(min(p[1] for p in r[0]) / max(gray_img.shape[0] / 2, 1)), min(p[0] for p in r[0])))
    text = re.sub(r'[^A-Z0-9]', '', ''.join(r[1].upper() for r in results))
    conf = sum(r[2] for r in results) / len(results) if results else 0.0
    return text, conf



@smart_inference_mode()
def run(
        weights=ROOT / 'best.pt',  # model path or triton URL
        source=ROOT / 'data/images',  # file/dir/URL/glob/screen/0(webcam)
        data=ROOT / 'yolov9/data/coco.yaml',  # dataset.yaml path
        imgsz=(640, 640),  # inference size (height, width)
        conf_thres=0.25,  # confidence threshold
        iou_thres=0.45,  # NMS IOU threshold
        max_det=1000,  # maximum detections per image
        device='',  # cuda device, i.e. 0 or 0,1,2,3 or cpu
        view_img=False,  # show results
        save_txt=False,  # save results to *.txt
        save_conf=False,  # save confidences in --save-txt labels
        save_crop=False,  # save cropped prediction boxes
        nosave=False,  # do not save images/videos
        classes=None,  # filter by class: --class 0, or --class 0 2 3
        agnostic_nms=False,  # class-agnostic NMS
        augment=False,  # augmented inference
        visualize=False,  # visualize features
        update=False,  # update all models
        project=ROOT / 'runs/detect',  # save results to project/name
        name='exp',  # save results to project/name
        exist_ok=False,  # existing project/name ok, do not increment
        line_thickness=3,  # bounding box thickness (pixels)
        hide_labels=False,  # hide labels
        hide_conf=False,  # hide confidences
        half=False,  # use FP16 half-precision inference
        dnn=False,  # use OpenCV DNN for ONNX inference
        vid_stride=1,  # video frame-rate stride
        save_csv=False,  # save recognised plate text to plates.csv
):
    source = str(source)
    save_img = not nosave and not source.endswith('.txt')  # save inference images
    is_file = Path(source).suffix[1:] in (IMG_FORMATS + VID_FORMATS)
    is_url = source.lower().startswith(('rtsp://', 'rtmp://', 'http://', 'https://'))
    webcam = source.isnumeric() or source.endswith('.txt') or (is_url and not is_file)
    screenshot = source.lower().startswith('screen')
    if is_url and is_file:
        source = check_file(source)  # download

    # Directories
    save_dir = increment_path(Path(project) / name, exist_ok=exist_ok)  # increment run
    (save_dir / 'labels' if save_txt else save_dir).mkdir(parents=True, exist_ok=True)  # make dir

    csv_rows = []

    # Load model
    device = select_device(device)
    model = DetectMultiBackend(weights, device=device, dnn=dnn, data=data, fp16=half)
    stride, names, pt = model.stride, model.names, model.pt
    imgsz = check_img_size(imgsz, s=stride)  # check image size

    # Dataloader
    bs = 1  # batch_size
    if webcam:
        view_img = check_imshow(warn=True)
        dataset = LoadStreams(source, img_size=imgsz, stride=stride, auto=pt, vid_stride=vid_stride)
        bs = len(dataset)
    elif screenshot:
        dataset = LoadScreenshots(source, img_size=imgsz, stride=stride, auto=pt)
    else:
        dataset = LoadImages(source, img_size=imgsz, stride=stride, auto=pt, vid_stride=vid_stride)
    vid_path, vid_writer = [None] * bs, [None] * bs

    # Run inference
    model.warmup(imgsz=(1 if pt or model.triton else bs, 3, *imgsz))  # warmup
    seen, windows, dt = 0, [], (Profile(), Profile(), Profile())
    for path, im, im0s, vid_cap, s in dataset:
        with dt[0]:
            im = torch.from_numpy(im).to(model.device)
            im = im.half() if model.fp16 else im.float()  # uint8 to fp16/32
            im /= 255  # 0 - 255 to 0.0 - 1.0
            if len(im.shape) == 3:
                im = im[None]  # expand for batch dim

        # Inference
        with dt[1]:
            visualize = increment_path(save_dir / Path(path).stem, mkdir=True) if visualize else False
            pred = model(im, augment=augment, visualize=visualize)

        # NMS
        with dt[2]:
            pred = pred[0][1] if isinstance(pred[0], list) else pred[0]
            pred = non_max_suppression(pred, conf_thres, iou_thres, classes, agnostic_nms, max_det=max_det)

        # Second-stage classifier (optional)
        # pred = utils.general.apply_classifier(pred, classifier_model, im, im0s)

        # Process predictions
        for i, det in enumerate(pred):  # per image
            seen += 1
            if webcam:  # batch_size >= 1
                p, im0, frame = path[i], im0s[i].copy(), dataset.count
                s += f'{i}: '
            else:
                p, im0, frame = path, im0s.copy(), getattr(dataset, 'frame', 0)

            p = Path(p)  # to Path
            save_path = str(save_dir / p.name)  # im.jpg
            txt_path = str(save_dir / 'labels' / p.stem) + ('' if dataset.mode == 'image' else f'_{frame}')  # im.txt
            s += '%gx%g ' % im.shape[2:]  # print string
            gn = torch.tensor(im0.shape)[[1, 0, 1, 0]]  # normalization gain whwh
            imc = im0.copy() if save_crop else im0  # for save_crop
            annotator = Annotator(im0, line_width=line_thickness, example=str(names))
            if len(det):
                # Rescale boxes from img_size to im0 size
                det[:, :4] = scale_boxes(im.shape[2:], det[:, :4], im0.shape).round()

                # Print results
                for c in det[:, 5].unique():
                    n = (det[:, 5] == c).sum()  # detections per class
                    s += f"{n} {names[int(c)]}{'s' * (n > 1)}, "  # add to string

                # Write results
                for *xyxy, conf, cls in reversed(det):
                    if save_txt:  # Write to file
                        xywh = (xyxy2xywh(torch.tensor(xyxy).view(1, 4)) / gn).view(-1).tolist()  # normalized xywh
                        line = (cls, *xywh, conf) if save_conf else (cls, *xywh)  # label format
                        with open(f'{txt_path}.txt', 'a') as f:
                            f.write(('%g ' * len(line)).rstrip() % line + '\n')

                    c = int(cls)  # integer class
                    text_ocr, ocr_conf = perform_ocr_on_image(im0, xyxy)  # EasyOCR on the plate crop
                    if text_ocr:
                        s += f'[{text_ocr}] '
                    if save_csv:
                        csv_rows.append([p.name, frame, text_ocr, f'{ocr_conf:.3f}', f'{float(conf):.3f}',
                                         *[int(v) for v in xyxy]])

                    if save_img or save_crop or view_img:  # Add bbox to image
                        if hide_labels:
                            label = None
                        elif text_ocr:
                            label = text_ocr
                        else:
                            label = names[c] if hide_conf else f'{names[c]} {conf:.2f}'
                        annotator.box_label(xyxy, label, color=colors(c, True))


                    if save_crop:
                        save_one_box(xyxy, imc, file=save_dir / 'crops' / names[c] / f'{p.stem}.jpg', BGR=True)

            # Stream results
            im0 = annotator.result()
            if view_img:
                if platform.system() == 'Linux' and p not in windows:
                    windows.append(p)
                    cv2.namedWindow(str(p), cv2.WINDOW_NORMAL | cv2.WINDOW_KEEPRATIO)  # allow window resize (Linux)
                    cv2.resizeWindow(str(p), im0.shape[1], im0.shape[0])
                cv2.imshow(str(p), im0)
                cv2.waitKey(1)  # 1 millisecond

            # Save results (image with detections)
            if save_img:
                if dataset.mode == 'image':
                    cv2.imwrite(save_path, im0)
                else:  # 'video' or 'stream'
                    if vid_path[i] != save_path:  # new video
                        vid_path[i] = save_path
                        if isinstance(vid_writer[i], cv2.VideoWriter):
                            vid_writer[i].release()  # release previous video writer
                        if vid_cap:  # video
                            fps = vid_cap.get(cv2.CAP_PROP_FPS)
                            w = int(vid_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                            h = int(vid_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                        else:  # stream
                            fps, w, h = 30, im0.shape[1], im0.shape[0]
                        save_path = str(Path(save_path).with_suffix('.mp4'))  # force *.mp4 suffix on results videos
                        vid_writer[i] = cv2.VideoWriter(save_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
                    vid_writer[i].write(im0)

        # Print time (inference-only)
        LOGGER.info(f"{s}{'' if len(det) else '(no detections), '}{dt[1].dt * 1E3:.1f}ms")

    if isinstance(vid_writer[-1], cv2.VideoWriter):
        vid_writer[-1].release()  # flush the last video so the file is playable
    if save_csv:
        with open(save_dir / 'plates.csv', 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['source', 'frame', 'plate_text', 'ocr_conf', 'det_conf', 'x1', 'y1', 'x2', 'y2'])
            writer.writerows(csv_rows)
        LOGGER.info(f"{len(csv_rows)} plate readings saved to {save_dir / 'plates.csv'}")

    # Print results
    t = tuple(x.t / seen * 1E3 for x in dt)  # speeds per image
    LOGGER.info(f'Speed: %.1fms pre-process, %.1fms inference, %.1fms NMS per image at shape {(1, 3, *imgsz)}' % t)
    if save_txt or save_img:
        s = f"\n{len(list(save_dir.glob('labels/*.txt')))} labels saved to {save_dir / 'labels'}" if save_txt else ''
        LOGGER.info(f"Results saved to {colorstr('bold', save_dir)}{s}")
    if update:
        strip_optimizer(weights[0])  # update model (to fix SourceChangeWarning)


def parse_opt():
    parser = argparse.ArgumentParser()
    parser.add_argument('--weights', nargs='+', type=str, default=ROOT / 'best.pt', help='model path or triton URL')
    parser.add_argument('--source', type=str, default=ROOT / 'data/images', help='file/dir/URL/glob/screen/0(webcam)')
    parser.add_argument('--data', type=str, default=ROOT / 'yolov9/data/coco.yaml', help='(optional) dataset.yaml path')
    parser.add_argument('--imgsz', '--img', '--img-size', nargs='+', type=int, default=[640], help='inference size h,w')
    parser.add_argument('--conf-thres', type=float, default=0.25, help='confidence threshold')
    parser.add_argument('--iou-thres', type=float, default=0.45, help='NMS IoU threshold')
    parser.add_argument('--max-det', type=int, default=1000, help='maximum detections per image')
    parser.add_argument('--device', default='', help='cuda device, i.e. 0 or 0,1,2,3 or cpu')
    parser.add_argument('--view-img', action='store_true', help='show results')
    parser.add_argument('--save-txt', action='store_true', help='save results to *.txt')
    parser.add_argument('--save-conf', action='store_true', help='save confidences in --save-txt labels')
    parser.add_argument('--save-crop', action='store_true', help='save cropped prediction boxes')
    parser.add_argument('--nosave', action='store_true', help='do not save images/videos')
    parser.add_argument('--classes', nargs='+', type=int, help='filter by class: --classes 0, or --classes 0 2 3')
    parser.add_argument('--agnostic-nms', action='store_true', help='class-agnostic NMS')
    parser.add_argument('--augment', action='store_true', help='augmented inference')
    parser.add_argument('--visualize', action='store_true', help='visualize features')
    parser.add_argument('--update', action='store_true', help='update all models')
    parser.add_argument('--project', default=ROOT / 'runs/detect', help='save results to project/name')
    parser.add_argument('--name', default='exp', help='save results to project/name')
    parser.add_argument('--exist-ok', action='store_true', help='existing project/name ok, do not increment')
    parser.add_argument('--line-thickness', default=3, type=int, help='bounding box thickness (pixels)')
    parser.add_argument('--hide-labels', default=False, action='store_true', help='hide labels')
    parser.add_argument('--hide-conf', default=False, action='store_true', help='hide confidences')
    parser.add_argument('--half', action='store_true', help='use FP16 half-precision inference')
    parser.add_argument('--dnn', action='store_true', help='use OpenCV DNN for ONNX inference')
    parser.add_argument('--vid-stride', type=int, default=1, help='video frame-rate stride')
    parser.add_argument('--save-csv', action='store_true', help='save recognised plate text to plates.csv')
    opt = parser.parse_args()
    opt.imgsz *= 2 if len(opt.imgsz) == 1 else 1  # expand
    print_args(vars(opt))
    return opt


def main(opt):
    check_requirements(exclude=('tensorboard', 'thop'))
    run(**vars(opt))


if __name__ == "__main__":
    opt = parse_opt()
    main(opt)

## 6. Train

In [ ]:
rc = sh(f'timeout {TRAIN_TIMEOUT} python train.py --batch 16 --epochs {EPOCHS} --img {IMG} --device {DEVICE} '
        f'--freeze {FREEZE} --min-items 0 --close-mosaic 5 --workers {min(8, CPUS)} --data {data_yaml} '
        f'--weights {W}/weights/gelan-c.pt --cfg models/detect/gelan-c.yaml --hyp data/hyps/hyp.scratch-high.yaml '
        f'--name plates --exist-ok', cwd=Y9, check=False)
print('train exit code', rc, '(124 = stopped by timeout, best.pt so far is kept)')

exp = f'{Y9}/runs/train/plates'
best = f'{exp}/weights/best.pt'
# A run stopped by the timeout leaves the optimizer state in best.pt (~4x bigger), so always strip it
sh(f'python -c "from utils.general import strip_optimizer; strip_optimizer(\'{best}\')"', cwd=Y9)
shutil.copy(best, f'{OUT}/best.pt')
for name in ('results.png', 'results.csv', 'confusion_matrix.png', 'F1_curve.png', 'PR_curve.png',
             'val_batch0_pred.jpg', 'val_batch0_labels.jpg', 'labels.jpg'):
    if os.path.exists(f'{exp}/{name}'):
        shutil.copy(f'{exp}/{name}', f'{OUT}/{name}')

In [ ]:
from IPython.display import Image, display
display(Image(f'{OUT}/results.png', width=1000))

## 7. Validate

In [ ]:
sh(f'python val.py --img {IMG} --batch 32 --conf 0.001 --iou 0.7 --device {DEVICE} --data {data_yaml} '
   f'--weights {best} --name plates_val --exist-ok 2>&1 | tee {OUT}/val_log.txt', cwd=Y9)
display(Image(f'{OUT}/val_batch0_pred.jpg', width=900))

## 8. ANPR on test images

In [ ]:
random.seed(0)
val_imgs = sorted(glob.glob(f'{root}/images/val/*'))
os.makedirs(f'{W}/samples', exist_ok=True)
for p in random.sample(val_imgs, min(12, len(val_imgs))):
    shutil.copy(p, f'{W}/samples/')
sh(f'python anpr.py --weights {best} --source {W}/samples --conf-thres 0.35 --device {DEVICE} --save-csv '
   f'--name anpr_images --exist-ok', cwd=Y9)
shutil.copytree(f'{Y9}/runs/detect/anpr_images', f'{OUT}/anpr_images', dirs_exist_ok=True)
for p in sorted(glob.glob(f'{OUT}/anpr_images/*.jpg'))[:4]:
    display(Image(p, width=600))

## 9. ANPR on a traffic video

In [ ]:
video = find_input('traffic_video_original.mp4')
sh(f'ffmpeg -y -loglevel error -t {VIDEO_SECONDS} -i {video} -an -c:v libx264 -preset fast {W}/traffic_clip.mp4')
sh(f'python anpr.py --weights {best} --source {W}/traffic_clip.mp4 --conf-thres 0.3 --device {DEVICE} --save-csv '
   f'--name anpr_video --exist-ok', cwd=Y9)
raw = glob.glob(f'{Y9}/runs/detect/anpr_video/*.mp4')[0]
sh(f'ffmpeg -y -loglevel error -i {raw} -c:v libx264 -pix_fmt yuv420p -crf 26 -preset fast -movflags +faststart '
   f'-vf "scale=1280:-2" {OUT}/anpr_demo.mp4')
sh(f'ffmpeg -y -loglevel error -ss 2 -t 8 -i {raw} -vf "fps=8,scale=720:-1:flags=lanczos,split[a][b];'
   f'[a]palettegen=max_colors=128[p];[b][p]paletteuse" {OUT}/anpr_demo.gif')
shutil.copy(f'{Y9}/runs/detect/anpr_video/plates.csv', f'{OUT}/video_plates.csv')
display(Image(f'{OUT}/anpr_demo.gif'))

## 10. Summary

In [ ]:
import pandas as pd

res = pd.read_csv(f'{exp}/results.csv')
res.columns = [c.strip() for c in res.columns]
vp = pd.read_csv(f'{OUT}/video_plates.csv')
summary = {
    'device': 'cuda' if GPU else f'cpu x{CPUS}',
    'epochs_planned': EPOCHS,
    'epochs_completed': int(len(res)),
    'img_size': IMG,
    'frozen_layers': FREEZE,
    'final_epoch_metrics': {k: float(v) for k, v in res.iloc[-1].items() if k.startswith('metrics')},
    'best_map50_95': float(res[[c for c in res.columns if 'mAP_0.5:0.95' in c][0]].max()),
    'video_detections': int(len(vp)),
    'video_top_plates': vp[vp.plate_text.fillna('').str.len() >= 5].plate_text.value_counts().head(15).to_dict(),
}
json.dump(summary, open(f'{OUT}/summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# Keep only the results (Kaggle saves everything under /kaggle/working as the notebook output)
for d in (Y9, f'{W}/weights', f'{W}/samples', f'{W}/subset', f'{W}/input'):
    shutil.rmtree(d, ignore_errors=True)
for f in glob.glob(f'{W}/*.mp4') + glob.glob(f'{W}/*.yaml'):
    os.remove(f)
if ON_COLAB:
    shutil.make_archive(f'{W}/anpr_results', 'zip', OUT)
    print('Download /content/anpr_results.zip from the Files panel')